<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/fine_tuning_patent_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install necessary packages to finetune datasets with SFT

In [ ]:
#!pip install -U transformers accelerate datasets bertviz umap-learn seaborn openpyxl --upgrade

In [ ]:
import os
os.environ['TORCH_USE_CUDA_DSA'] = '1'

In [ ]:
import warnings
warnings.filterwarnings('ignore')

Login to HuggingFace. I have stored my creds in colab secrets

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

splits = {'train': 'patent/train-00000-of-00001.parquet', 'validation': 'patent/validation-00000-of-00001.parquet', 'test': 'patent/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/ccdv/patent-classification/" + splits["train"])

In [ ]:
df.dtypes

In [ ]:
df = df.rename(columns={'label': 'labels'})

In [ ]:
#Bin the categories

freq_df = df.groupby('labels').size().reset_index(name='count')

freq_df
#min(freq_df['count']) , max(freq_df['count'])



# Plot the frequency of classes

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
label_counts = df['labels'].value_counts(ascending=True)
label_counts.plot.barh()
plt.title('Frequency of Classes')
plt.show()

Data Loader and Splitting the Data

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3,stratify=df['labels']) #can play around with the size of the test datset

test, validation = train_test_split(test, test_size=1/3, stratify=test['labels'])


In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train, preserve_index=False),
        "test": Dataset.from_pandas(test, preserve_index=False),
        "validation": Dataset.from_pandas(validation, preserve_index=False),
    }
)

dataset

# Data Tokenization

In [ ]:
from transformers import AutoTokenizer

text = "Riddhiman Sherlekar is an expert in AI"

model_checkpoint = "distilbert-base-uncased"
distilbert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
distilbert_tokens = distilbert_tokenizer.tokenize(text)

In [ ]:
def tokenize(batch):
  temp = distilbert_tokenizer(batch['text'], padding=True, truncation=True)
  return temp


print(tokenize(dataset['train'][:3]))

In [ ]:
encoded_dataset = dataset.map(
    tokenize,
    batched=True,
    batch_size=1000,  # <--- IMPORTANT: Set a specific batch size here
    num_proc=4, # <--- Optional: Uncomment and set if you want parallel processing
    remove_columns=['text'] # Optional: Remove the original text column to save memory
)

# Model Building

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoConfig

import torch

label2id  = {'Physics': 6, 'Human Necessities': 0, 'Electricity': 7, 'Performing Operations': 1, 'General tagging': 8, 'Chemistry': 2, 'Fixed Constructions': 4, 'Textiles': 3 , 'Mechanical Engineering':5}

id2label  =  {6: 'Physics', 0: 'Human Necessities', 7: 'Electricity', 1: 'Performing Operations', 8: 'General tagging', 2: 'Chemistry', 4: 'Fixed Constructions', 3: 'Textiles', 5:'Mechanical Engineering'}

In [ ]:
len(label2id)
len(id2label)

In [ ]:
model_checkpoint = "distilbert-base-uncased"

num_labels = len(label2id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = AutoConfig.from_pretrained(model_checkpoint, label2id=label2id, id2label=id2label)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint,config=config).to(device)

In [ ]:
device

# Model Fine Tuning

The first thing we need is a pretrained BERT like model
THe only slight modification is that we use the AutoModelForSequenceClassification model instead of AutoModel

The difference is that the AutoModelForSequenceClassification model has a classification head on top of pretrained model outputs, which can be easily trained with the base model.

Evaluate

In [ ]:
#!pip install evaluate

In [ ]:
import evaluate
import numpy as np


accuracy = evaluate.load("accuracy")

def compute_metrics_evaluate(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  return accuracy.compute(predictions=predictions, references=labels)

  accuracy = evaluate.load("accuracy")

def compute_metrics_evaluate(eval_pred):
    predictions, labels = eval_pred

    # --- DEBUGGING STEP 1: Check raw predictions ---
    print(f"Predictions type: {type(predictions)}, shape: {predictions.shape}")
    # Convert predictions to a PyTorch tensor temporarily to use .isnan()/.isinf()
    # If predictions are already numpy, convert to tensor:
    if isinstance(predictions, np.ndarray):
        predictions_tensor = torch.from_numpy(predictions)
    else: # Assume it's already a tensor if not numpy
        predictions_tensor = predictions

    if torch.isnan(predictions_tensor).any():
        print("\n!!! NaN detected in raw predictions !!!")
        print("Raw Predictions (first 5 rows):\n", predictions[:5])
        # Optionally, save predictions to a file for later inspection
        # np.save('nan_predictions.npy', predictions)
        raise ValueError("NaNs found in model predictions during evaluation!")

    if torch.isinf(predictions_tensor).any():
        print("\n!!! Inf detected in raw predictions !!!")
        print("Raw Predictions (first 5 rows):\n", predictions[:5])
        raise ValueError("Infs found in model predictions during evaluation!")

    # --- DEBUGGING STEP 2: Check labels ---
    print(f"Labels type: {type(labels)}, shape: {labels.shape}")
    if isinstance(labels, np.ndarray):
        labels_tensor = torch.from_numpy(labels)
    else:
        labels_tensor = labels

    if torch.isnan(labels_tensor).any():
        print("\n!!! NaN detected in labels !!!")
        raise ValueError("NaNs found in labels during evaluation!")
    if torch.isinf(labels_tensor).any():
        print("\n!!! Inf detected in labels !!!")
        raise ValueError("Infs found in labels during evaluation!")


    # Proceed with your logic if predictions and labels are clean
    predictions = np.argmax(predictions, axis=1)

    # --- DEBUGGING STEP 3: Check argmax output ---
    if np.isnan(predictions).any():
        print("\n!!! NaN detected after np.argmax in predictions !!!")
        raise ValueError("NaNs found after argmax!")
    if np.isinf(predictions).any():
        print("\n!!! Inf detected after np.argmax in predictions !!!")
        raise ValueError("Infs found after argmax!")

    # Ensure predictions and references have compatible types and values for the metric
    print(f"Predictions after argmax (first 10): {predictions[:10]}")
    print(f"Labels (first 10): {labels[:10]}")


    return accuracy.compute(predictions=predictions, references=labels)



In [ ]:
from transformers import TrainingArguments

#In the latest transformer version evaluation_strategy was replaced by eval_strategy

batch_size = 32
training_dir = "train_dir"

training_args = TrainingArguments(
    output_dir = training_dir,
    overwrite_output_dir = True,
    num_train_epochs = 2,
    learning_rate = 2e-5,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    weight_decay = 0.01,
    # Corrected argument name
    eval_strategy = 'epoch',
    # It's also common to set save_strategy to match eval_strategy
    save_strategy = 'epoch',
    max_grad_norm=1.0,
    # Optionally, specify how often to save checkpoints based on the strategy
    save_steps = 500 # This would save every 500 steps if save_strategy was 'steps'
                     # For 'epoch', it typically saves at the end of each epoch by default.
                     # You can omit save_steps if you just want to save per epoch
)

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    compute_metrics = compute_metrics_evaluate,
    train_dataset = encoded_dataset['train'],
    eval_dataset= encoded_dataset['validation'],
    tokenizer=distilbert_tokenizer,

)

In [ ]:
import torch
torch.autograd.set_detect_anomaly(True)

# ... your trainer setup ...
trainer.train()

SAve the model

In [ ]:
trainer.save_model("rsher60/patent_classification")

# Model Evaluation

In [ ]:
preds_output = trainer.predict(encoded_dataset['test'])

In [ ]:
preds_output.metrics

In [ ]:
y_pred = np.argmax(preds_output.predictions, axis = 1)
y_true = encoded_dataset['test'][:]['labels']

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=list(label2id)))

In [ ]:
# Assuming encoded_dataset is a DatasetDict from Hugging Face datasets library
# and it contains a 'labels' column.

import numpy as np

# Convert labels to a flat list or numpy array to find max
train_labels = encoded_dataset['train']['labels']
max_train_label = np.max(train_labels)

validation_labels = encoded_dataset['validation']['labels']
test_labels = encoded_dataset['test']['labels']
max_validation_label = np.max(validation_labels)
max_test_label = np.max(test_labels)

print(f"Maximum label in training dataset: {max_train_label}")
print(f"Maximum label in validation dataset: {max_validation_label}")
print(f"Maximum label in testing dataset: {max_test_label}")

# The true number of classes will be (max_label + 1)
# Make sure to take the overall maximum if train and val have different maxes
actual_num_classes = max(max_train_label, max_validation_label) + 1
print(f"Actual number of classes in your dataset: {actual_num_classes}")

# Verify all labels are non-negative (0 or greater)
min_train_label = np.min(train_labels)
min_validation_label = np.min(validation_labels)
print(f"Minimum label in training dataset: {min_train_label}")
print(f"Minimum label in validation dataset: {min_validation_label}")

if min_train_label < 0 or min_validation_label < 0:
    print("WARNING: Labels should be non-negative (0 or greater).")